# 🇹🇳 Tunisian NER — Fine-tuning CamemBERT

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

CamemBERT is trained on 138GB of French text — perfect for Tunisian French NER.

## ✅ Step 1 — Install dependencies

In [1]:
!pip install -q transformers datasets seqeval accelerate huggingface_hub safetensors
print('✅ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
✅ Dependencies installed


## ✅ Step 2 — Upload your data files

In [2]:
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print('Upload your train.conll, dev.conll and test.conll:')
uploaded = files.upload()

for fname in uploaded:
    os.rename(fname, f'data/{fname}')
    print(f'  ✅ {fname} uploaded')

Upload your train.conll, dev.conll and test.conll:


Saving dev.conll to dev.conll
Saving test.conll to test.conll
Saving train.conll to train.conll
  ✅ dev.conll uploaded
  ✅ test.conll uploaded
  ✅ train.conll uploaded


## ✅ Step 3 — Load & parse CoNLL data

In [3]:
LABEL_LIST = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}

def load_conll(filepath):
    all_tokens, all_labels = [], []
    tokens, labels = [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == '':
                if tokens:
                    all_tokens.append(tokens)
                    all_labels.append(labels)
                    tokens, labels = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    token, label = parts
                    label = label if label in LABEL2ID else 'O'
                    tokens.append(token.strip())
                    labels.append(LABEL2ID[label])
    if tokens:
        all_tokens.append(tokens)
        all_labels.append(labels)
    return all_tokens, all_labels

train_tokens, train_labels = load_conll('data/train.conll')
dev_tokens,   dev_labels   = load_conll('data/dev.conll')
test_tokens,  test_labels  = load_conll('data/test.conll')

print(f'✅ Loaded:')
print(f'   train : {len(train_tokens):,} sentences')
print(f'   dev   : {len(dev_tokens):,} sentences')
print(f'   test  : {len(test_tokens):,} sentences')

✅ Loaded:
   train : 36,737 sentences
   dev   : 4,592 sentences
   test  : 4,593 sentences


## ✅ Step 4 — Load CamemBERT

In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

MODEL_NAME = 'camembert-base'

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print('Loading CamemBERT model...')
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = model.to(device)

print(f'\n✅ CamemBERT ready on {device}')
print(f'   Parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'   Labels     : {LABEL_LIST}')

Loading tokenizer...


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

Loading CamemBERT model...


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ CamemBERT ready on cuda
   Parameters : 110,036,743
   Labels     : ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG']


## ✅ Step 5 — Tokenize & align labels

In [5]:
from datasets import Dataset

def tokenize_and_align(tokens_list, labels_list):
    tokenized = tokenizer(
        tokens_list,
        truncation=True,
        max_length=128,
        is_split_into_words=True,
        padding=False,
    )
    aligned_labels = []
    for i, label_ids in enumerate(labels_list):
        word_ids     = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        row          = []
        for word_id in word_ids:
            if word_id is None:
                row.append(-100)
            elif word_id != prev_word_id:
                row.append(label_ids[word_id])
            else:
                row.append(-100)
            prev_word_id = word_id
        aligned_labels.append(row)
    tokenized['labels'] = aligned_labels
    return tokenized

def build_dataset(tokens_list, labels_list):
    ds = Dataset.from_dict({'tokens': tokens_list, 'ner_tags': labels_list})
    return ds.map(
        lambda x: tokenize_and_align(x['tokens'], x['ner_tags']),
        batched=True,
        remove_columns=['tokens', 'ner_tags']
    )

print('Tokenizing...')
train_ds = build_dataset(train_tokens, train_labels)
dev_ds   = build_dataset(dev_tokens,   dev_labels)
test_ds  = build_dataset(test_tokens,  test_labels)

print(f'✅ Done — {len(train_ds):,} train examples')

Tokenizing...


Map:   0%|          | 0/36737 [00:00<?, ? examples/s]

Map:   0%|          | 0/4592 [00:00<?, ? examples/s]

Map:   0%|          | 0/4593 [00:00<?, ? examples/s]

✅ Done — 36,737 train examples


## ✅ Step 6 — Define metrics

In [6]:
import numpy as np
from seqeval.metrics import f1_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        true_labels.append([ID2LABEL[l] for l in label_row if l != -100])
        true_preds.append([ID2LABEL[p] for p, l in zip(pred_row, label_row) if l != -100])
    report = classification_report(true_labels, true_preds, output_dict=True, zero_division=0)
    return {
        'f1'     : f1_score(true_labels, true_preds, zero_division=0),
        'f1_per' : report.get('PER', {}).get('f1-score', 0.0),
        'f1_loc' : report.get('LOC', {}).get('f1-score', 0.0),
        'f1_org' : report.get('ORG', {}).get('f1-score', 0.0),
    }

print('✅ Metrics ready')

✅ Metrics ready


## ✅ Step 7 — Train

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

training_args = TrainingArguments(
    output_dir                  = './tun-ner-camembert',
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 3e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    save_total_limit            = 2,
    logging_steps               = 50,
    fp16                        = True,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = dev_ds,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

print('🚀 Starting training...')
print(f'   Model      : camembert-base')
print(f'   Epochs     : {training_args.num_train_epochs}')
print(f'   Batch size : {training_args.per_device_train_batch_size}')
print(f'   Device     : {device}\n')

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Starting training...
   Model      : camembert-base
   Epochs     : 10
   Batch size : 16
   Device     : cuda



Epoch,Training Loss,Validation Loss,F1,F1 Per,F1 Loc,F1 Org
1,0.292349,0.208108,0.884067,0.858257,0.918956,0.829062
2,0.131493,0.086448,0.944933,0.937952,0.963043,0.909918
3,0.081205,0.048573,0.968905,0.960712,0.978423,0.954111
4,0.036698,0.044821,0.973830,0.967213,0.980697,0.963467
5,0.018665,0.033381,0.983206,0.978140,0.988121,0.976168
6,0.018614,0.032237,0.983958,0.977105,0.988484,0.979391
7,0.007187,0.028879,0.986137,0.980015,0.990003,0.982413
8,0.013838,0.027671,0.987332,0.981859,0.990543,0.984558
9,0.008378,0.028210,0.987782,0.980015,0.992292,0.983970
10,0.004776,0.028283,0.988006,0.981118,0.991888,0.984891


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=22970, training_loss=0.10573718414638268, metrics={'train_runtime': 1164.485, 'train_samples_per_second': 315.479, 'train_steps_per_second': 19.725, 'total_flos': 1.2239862794374776e+16, 'train_loss': 0.10573718414638268, 'epoch': 10.0})

## ✅ Step 8 — Evaluate on test set

In [8]:
print('🧪 Evaluating on test set...\n')
results = trainer.evaluate(test_ds)

print(f'\n📊 Final Results')
print(f'  Overall F1 : {results.get("eval_f1", 0):.4f}')
print(f'  PER F1     : {results.get("eval_f1_per", 0):.4f}')
print(f'  LOC F1     : {results.get("eval_f1_loc", 0):.4f}')
print(f'  ORG F1     : {results.get("eval_f1_org", 0):.4f}')

🧪 Evaluating on test set...




📊 Final Results
  Overall F1 : 0.9904
  PER F1     : 0.9892
  LOC F1     : 0.9929
  ORG F1     : 0.9859


## ✅ Step 9 — Upload to HuggingFace Hub

In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [10]:
HF_USERNAME = 'NourBesrour'
REPO_NAME   = 'tun-ner-camembert'

trainer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')

print(f'\n🎉 Model uploaded!')
print(f'   https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...membert/model.safetensors:   0%|          |  552kB /  440MB            

  ...membert/training_args.bin:   1%|1         |  76.0B / 5.20kB            

README.md:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.



🎉 Model uploaded!
   https://huggingface.co/NourBesrour/tun-ner-camembert


## ✅ Step 10 — Quick test in Colab

In [11]:
from transformers import pipeline

ner = pipeline('ner', model=model, tokenizer=tokenizer,
               aggregation_strategy='simple', device=0)

test_sentences = [
    'Ahmed Karray dirige la STEG à Tunis.',
    'Le ministre Samir Saied a visité Sfax hier.',
    'Tunisair a annoncé de nouveaux vols vers Paris.',
]

for sent in test_sentences:
    print(f'\n📝 {sent}')
    entities = ner(sent)
    for ent in entities:
        print(f'   → {ent["word"]:<25} {ent["entity_group"]}  ({ent["score"]:.2f})')


📝 Ahmed Karray dirige la STEG à Tunis.
   → Ahmed Karray              PER  (1.00)
   → STEG                      ORG  (1.00)
   → Tunis                     LOC  (1.00)

📝 Le ministre Samir Saied a visité Sfax hier.
   → Samir Saied               PER  (1.00)
   → S                         LOC  (1.00)
   → fa                        LOC  (1.00)
   → x                         LOC  (0.84)

📝 Tunisair a annoncé de nouveaux vols vers Paris.
   → Tunisair                  ORG  (0.99)
   → Paris                     LOC  (1.00)
